In [ ]:
import gc
import itertools
import os
import random
import sys
from dataclasses import dataclass
from functools import partial
from pathlib import Path
from typing import Any, Literal, TypeAlias

import circuitsvis as cv
import einops
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import requests
import torch as t
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download
from IPython.display import HTML, IFrame, display
from jaxtyping import Float, Int
from openai import OpenAI
from rich import print as rprint
from rich.table import Table
from sae_lens import (
    SAE,
    ActivationsStore,
    GatedTrainingSAEConfig,
    HookedSAETransformer,
    LanguageModelSAERunnerConfig,
    LoggingConfig,
)
from sae_lens.loading.pretrained_saes_directory import get_pretrained_saes_directory
from sae_vis import SaeVisConfig, SaeVisData, SaeVisLayoutConfig
from tabulate import tabulate
from torch import Tensor
from tqdm.auto import tqdm
from transformer_lens import ActivationCache
from transformer_lens.hook_points import HookPoint
from transformer_lens.utils import get_act_name, test_prompt

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")


def _get_hook_layer(sae: SAE) -> int:
  """Extract the layer number from an SAE's hook name (e.g. 'blocks.7.hook_resid_pre' → 7)."""
  return int(sae.cfg.metadata.hook_name.split(".")[1])

def display_dashboard(
  sae_release="goodfire-llama-3.1-8b-instruct",
  sae_id="blocks.7.hook_resid_pre",
  latent_idx=0,
  width=800,
  height=600,
):
  release = get_pretrained_saes_directory()[sae_release]
  neuronpedia_id = release.neuronpedia_id[sae_id]

  url = f"https://neuronpedia.org/{neuronpedia_id}/{latent_idx}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"

  print(url)
  display(IFrame(url, width=width, height=height))

In [4]:
metadata_rows = [
    [data.model, data.release, data.repo_id, len(data.saes_map)] for data in get_pretrained_saes_directory().values()
]


print(
    tabulate(
        sorted(metadata_rows, key=lambda x: x[0]),
        headers=["model", "release", "repo_id", "n_saes"],
        tablefmt="simple_outline",
    )
)


┌──────────────────────────────────────────┬─────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────┬──────────┐
│ model                                    │ release                                             │ repo_id                                                   │   n_saes │
├──────────────────────────────────────────┼─────────────────────────────────────────────────────┼───────────────────────────────────────────────────────────┼──────────┤
│ Qwen/Qwen2.5-7B-Instruct                 │ qwen2.5-7b-instruct-andyrdt                         │ andyrdt/saes-qwen2.5-7b-instruct                          │        7 │
│ deepseek-ai/DeepSeek-R1-Distill-Llama-8B │ llama_scope_r1_distill                              │ fnlp/Llama-Scope-R1-Distill                               │       96 │
│ deepseek-ai/DeepSeek-R1-Distill-Llama-8B │ deepseek-r1-distill-llama-8b-qresearch              │ qresearch/DeepSeek-R1-Distill-Llama-8B-SAE-l19     

In [ ]:
gpt2: HookedSAETransformer = HookedSAETransformer.from_pretrained("gpt2-small", device=device)
gpt2_sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.7.hook_resid_pre",
    device=str(device),
)

In [11]:
release = get_pretrained_saes_directory()["goodfire-llama-3.3-70b-instruct"]


In [12]:
data = [[id, path, release.neuronpedia_id[id]] for id, path in release.saes_map.items()]

print(
    tabulate(
        data,
        headers=["SAE id", "SAE path (HuggingFace)", "Neuronpedia ID"],
        tablefmt="simple_outline",
    )
)

┌──────────┬───────────────────────────────────┬──────────────────────────────────┐
│ SAE id   │ SAE path (HuggingFace)            │ Neuronpedia ID                   │
├──────────┼───────────────────────────────────┼──────────────────────────────────┤
│ layer_50 │ Llama-3.3-70B-Instruct-SAE-l50.pt │ llama3.3-70b-it/50-resid-post-gf │
└──────────┴───────────────────────────────────┴──────────────────────────────────┘
